# ⚡ Istovremeni agentni radni tokovi s Microsoft Foundry (Python)

## 📋 Napredni vodič za paralelnu obradu

Ovaj bilježnik prikazuje **uzorke istovremenih radnih tokova** koristeći Microsoft Agent Framework. Naučit ćete kako izgraditi visokoučinkovite, paralelne radne tokove u kojima više AI agenata radi istovremeno, što dramatično poboljšava kapacitet obrade i omogućuje sofisticirane višestruke poslovne procese s više niti.

> **Napomena o migraciji:** Ovaj primjer je prethodno koristio GitHub Models. GitHub Models je zastario (ukida se u srpnju 2026.), pa sada koristi **Microsoft Foundry** putem `FoundryChatClient`, koji cilja Azure OpenAI **Responses API**.

## 🎯 Ciljevi učenja

### 🚀 **Osnove istovremene obrade**
- **Paralelno izvršavanje agenata**: Pokrenite više agenata istovremeno za maksimalnu učinkovitost
- **Orkestracija radnih tokova**: Koordinirajte istovremene operacije uz održavanje konzistentnosti podataka
- **Optimizacija performansi**: Postignite značajno ubrzanje kroz paralelnu obradu
- **Upravljanje resursima**: Učinkovito koristite resurse AI modela u istovremenim operacijama

### 🏗️ **Napredni obrasci konkurentnosti**
- **Fork-Join obrada**: Podijelite rad na više agenata i spojite rezultate
- **Pipeline paralelizam**: Preklapajuće faze izvođenja za kontinuirani kapacitet obrade
- **Raspodjela opterećenja**: Ravnomjerno raspodijelite rad između dostupnih resursa agenata
- **Točke sinkronizacije**: Koordinirajte istovremene agente u ključnim fazama radnog toka

### 🏢 **Istovremene poslovne aplikacije**
- **Obrada velikog volumena dokumenata**: Obradite više dokumenata istovremeno
- **Analiza sadržaja u stvarnom vremenu**: Istovremena analiza dolaznih tokova podataka
- **Optimizacija obrade paketa**: Maksimizirajte kapacitet za velike operacije
- **Višestruka multimodalna analiza**: Paralelna obrada različitih vrsta sadržaja (tekst, slike, podaci)

## ⚙️ Preduvjeti i postavljanje

### 📦 **Potrebne ovisnosti**

Instalirajte Agent Framework s mogućnostima istovremenih radnih tokova:

```bash
pip install agent-framework -U
```

### 🔑 **Konfiguracija Microsoft Foundry**

Prijavite se koristeći Azure CLI (`az login`) kako bi `AzureCliCredential` mogao izvršiti autentifikaciju, zatim postavite detalje vašeg Microsoft Foundry projekta.

**Postavke okoline (.env datoteka):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

**Razmatranja istovremene obrade:**
- **Ograničenja stope**: Pratite ograničenja Azure OpenAI za istovremene zahtjeve
- **Korištenje resursa**: Uzmite u obzir upotrebu memorije i CPU-a s više istovremenih agenata
- **Rukovanje pogreškama**: Implementirajte robusni oporavak od pogrešaka za paralelne operacije

### 🏗️ **Arhitektura istovremenih radnih tokova**

```mermaid
graph TD
    A[Početak radnog procesa] --> B[Istovremeno izvršavanje]
    B --> C[Skup agenata 1]
    B --> D[Skup agenata 2]
    B --> E[Skup agenata 3]
    C --> F[Agregacija rezultata]
    D --> F
    E --> F
    F --> G[Završni izlaz]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Glavne prednosti:**
- **⚡ Performanse**: Značajno ubrzanje kroz paralelno izvođenje
- **📈 Skalabilnost**: Podnesite povećane obrate bez proporcionalnog povećanja trajanja
- **🔄 Učinkovitost**: Bolje iskorištavanje dostupnih računalnih resursa
- **🎯 Kapacitet obrade**: Obradite više posla u istom vremenskom razdoblju

## 🎨 **Obrasci dizajna istovremenih radnih tokova**

### 🔍 **Cjevovod za istraživanje i analizu**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Radni tok obrade podataka**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Cjevovod za stvaranje sadržaja**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Višefazna obrada**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Prednosti performansi za tvrtke**

### ⚡ **Optimizacija kapaciteta obrade**
- **Paralelno izvođenje**: Više agenata radi istovremeno
- **Korištenje resursa**: Maksimalna učinkovitost dostupnog kapaciteta AI modela
- **Smanjenje vremena**: Značajno smanjenje ukupnog vremena obrade
- **Skalabilna arhitektura**: Jednostavno dodavanje više istovremenih agenata po potrebi

### 🛡️ **Pouzdanost i otpornost**
- **Otpornost na kvarove**: Pojedinačni kvarovi agenata ne zaustavljaju cijeli radni tok
- **Izolacija pogrešaka**: Problemi u jednoj konkurentnoj grani ne utječu na ostale
- **Graceful Degradation**: Sustav nastavlja rad čak i uz smanjeni kapacitet agenata
- **Mehanizmi oporavka**: Automatsko ponovno pokušavanje i rukovanje pogreškama za neuspjele operacije

### 📊 **Praćenje i vidljivost**
- **Praćenje istovremenog izvođenja**: Nadzirite napredak svih paralelnih operacija
- **Metričke performansi**: Mjerite ubrzanje i dobitke u učinkovitosti
- **Analiza korištenja resursa**: Optimizirajte raspodjelu istovremenih agenata
- **Identifikacija uskih grla**: Pronađite i uklonite ograničenja performansi

Izgradimo visokoučinkovite istovremene AI radne tokove! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
